In [ ]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

1 Introduction

OSM-GeoJSON to RDF Knowledge Graph

This notebook demonstrates how OpenStreetMap (OSM) vector data (ways) can be transformed into an RDF knowledge graph using Python.

It covers the full workflow from loading GeoJSON data with GeoPandas to generating RDF triples with rdflib, including GeoSPARQL geometries and Turtle serialization.


* 1 Load GeoJSON 
* 2 Add column "length" to osm 
* 3 Prepare CRS
* 4 Build RDF Graph
* 5 Add GeoSPARQL Geometries
* 6 Export Turtle



In [ ]:
# Import packages
import pandas as pd
import geopandas as gpd
from pathlib import Path
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, SKOS
from rdflib.namespace import XSD

In [ ]:
# Namespaces
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")
GEO  = Namespace("http://www.opengis.net/ont/geosparql#")

In [ ]:
# Import Data

BASE_DIR = Path.cwd().parent  # eine Ebene nach oben

OUTPUT_DIR = BASE_DIR / "data" / "processed"

INPUT_FILE = BASE_DIR / "data" / "osm" / "ways_osm.geojson"

OUTPUT_FILE = BASE_DIR / "data" / "processed" / "osm_individuals_with_geom.ttl"



In [ ]:
# Set CRS to UTM for length calculation

gdf = gpd.read_file(INPUT_FILE)

if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(4326)

In [ ]:
# Set CRS to UTM for length calculation

gdf_utm = gdf.to_crs(25832)

# Add column "length" to gdf

gdf["length_m"] = round(gdf_utm.length, 2)



In [ ]:
# Initial Graph

g = Graph()

g.bind("agro", AGRO)
g.bind("geo", GEO)
g.bind("skos", SKOS)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

In [ ]:

# Untersuchungsgebiet/ test area "test_area_1"
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, XSD, SKOS
from rdflib.namespace import NamespaceManager
import pandas as pd

# Graph
g = Graph()

# Namespaces
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")
GEO = Namespace("http://www.opengis.net/ont/geosparql#")

g.bind("agro", AGRO)
g.bind("geo", GEO)
g.bind("skos", SKOS)

# Untersuchungsgebiet
test_area_uri = AGRO["test_area_1"]

# OSM Ways erzeugen
for idx, row in gdf.iterrows():

    way_uri  = AGRO[f"osm_way_{idx+1}"]
    geom_uri = AGRO[f"geom_osm_way_{idx+1}"]

    # Klasse
    g.add((way_uri, RDF.type, AGRO.OSMWay))

    # Label
    g.add((
        way_uri,
        SKOS.prefLabel,
        Literal(f"OSM Way {idx+1}", lang="en")
    ))

    # Geometrie-Klasse
    g.add((geom_uri, RDF.type, GEO.Geometry))

    # WKT-Geometrie
    g.add((
        geom_uri,
        GEO.asWKT,
        Literal(
            row.geometry.wkt,
            datatype=GEO.wktLiteral
        )
    ))

    # Verbindung Way zu Geometrie
    g.add((way_uri, GEO.hasGeometry, geom_uri))

    # Beziehung zum Untersuchungsgebiet
    g.add((way_uri, GEO.sfWithin, test_area_uri))

    # Spezialfelder separat behandeln
    # --------------------------------

    # OSM-ID
    if not pd.isna(row["id"]):

        g.add((
            way_uri,
            AGRO.osmWayId,
            Literal(str(row["id"]))
        ))

    # Länge
    if not pd.isna(row["length_m"]):

        g.add((
            way_uri,
            AGRO.lengthMeters,
            Literal(
                round(float(row["length_m"]), 2),
                datatype=XSD.decimal
            )
        ))

    # Allgemeine Attribute
    # --------------------------------

    for col in gdf.columns:

        # Bereits separat behandelte Felder überspringen
        if col in ["geometry", "id", "length_m"]:
            continue

        value = row[col]

        # Leere Werte überspringen
        if pd.isna(value):
            continue

        # Property-Name bereinigen
        pred = AGRO[
            col.replace(":", "_")
               .replace("@", "")
               .replace("-", "_")
        ]

        # Integer
        if str(value).isdigit():

            obj = Literal(
                int(value),
                datatype=XSD.integer
            )

        # Float
        else:
            try:
                float_val = float(value)

                obj = Literal(
                    float_val,
                    datatype=XSD.decimal
                )

            # String
            except:
                obj = Literal(str(value))

        # Triple hinzufügen
        g.add((way_uri, pred, obj))

In [ ]:
gdf[Index(['id', '@id', 'cycleway:right', 'cycleway:right:bicycle',
       'cycleway:right:buffer', 'cycleway:right:foot',
       'cycleway:right:segregated', 'cycleway:right:smoothness',
       'cycleway:right:surface', 'cycleway:right:width',
       ...
       'ramp:wheelchair', 'cycleway:both:traffic_sign',
       'sidewalk:left:bicycle', 'cycleway:both:bicycle',
       'cycleway:both:segregated', 'cycleway:right:oneway', 'stroller',
       'maxheight', 'geometry', 'length_m'],
      dtype='str', length=140)]

In [ ]:
# create and save RDF File

g.serialize(destination=OUTPUT_FILE, format="turtle")

print("Features:", len(gdf))
print("Triples:", len(g))
print("Saved:", OUTPUT_FILE)